# Epistemic Extension Slice

Journey purpose: prove the first extension-local epistemic slice over accepted
candidate assertions.

Notebook use: `proof`

This deep-dive notebook shows the narrow Phase 9 path: confidence assessment
plus supersession, both attached to accepted candidate assertions and kept out
of the base review schema.

Related docs, code, tests, and evidence:

- `docs/plans/0001_successor_roadmap.md`
- `docs/plans/0003_phase9_epistemic_shape.md`
- `docs/adr/0009-start-epistemic-extension-with-confidence-and-supersession-over-accepted-candidates.md`
- `src/onto_canon6/extensions/epistemic/`
- `src/onto_canon6/surfaces/epistemic_report.py`
- `tests/extensions/test_epistemic_service.py`


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

SEARCH_ROOTS = [
    Path.cwd().resolve(),
    Path('~/projects/onto-canon6').expanduser(),
]

PROJECT_ROOT = None
for start in SEARCH_ROOTS:
    candidate = start
    while True:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    if PROJECT_ROOT is not None:
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate onto-canon6 repo root from notebook cwd or ~/projects/onto-canon6"
    )

for candidate_path in (
    PROJECT_ROOT / "src",
    PROJECT_ROOT.parent / "llm_client",
    Path('~/projects/llm_client').expanduser(),
):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

from pprint import pprint
from tempfile import TemporaryDirectory

from onto_canon6.extensions.epistemic import EpistemicService
from onto_canon6.pipeline import ReviewService
from onto_canon6.surfaces import EpistemicReportService


## Phase 1: Accepted Candidate Setup

Purpose: persist two accepted candidates that the epistemic extension can act
on through explicit seams.

Input -> output: `review_ready_candidates -> accepted_candidates`

Acceptance criteria:
- Two candidate assertions are persisted and accepted through the existing review workflow.
- The extension attaches only after review acceptance.
- The phase runs live.

Status: `proven`
Execution mode: `live`


In [2]:
workspace = TemporaryDirectory()
workspace_path = Path(workspace.name)
review_service = ReviewService(
    db_path=workspace_path / 'review.sqlite3',
    overlay_root=workspace_path / 'ontology_overlays',
    default_acceptance_policy='record_only',
)

first_submission = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'oc:epistemic_demo_original',
        'roles': {'subject': [{'entity_id': 'ent:campaign:alpha'}]},
    },
    profile_id='default',
    profile_version='1.0.0',
    submitted_by='analyst:notebook',
    source_kind='note',
    source_ref='notes/original.txt',
)
second_submission = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'oc:epistemic_demo_replacement',
        'roles': {'subject': [{'entity_id': 'ent:campaign:alpha'}]},
    },
    profile_id='default',
    profile_version='1.0.0',
    submitted_by='analyst:notebook',
    source_kind='note',
    source_ref='notes/replacement.txt',
)
first_candidate = review_service.review_candidate(
    candidate_id=first_submission.candidate.candidate_id,
    decision='accepted',
    actor_id='analyst:reviewer',
)
second_candidate = review_service.review_candidate(
    candidate_id=second_submission.candidate.candidate_id,
    decision='accepted',
    actor_id='analyst:reviewer',
)
pprint(first_candidate.model_dump())
pprint(second_candidate.model_dump())


{'candidate_id': 'cand_576cf7b9d22a4479a198dc67',
 'claim_text': None,
 'evidence_spans': (),
 'normalized_payload': {'predicate': 'oc:epistemic_demo_original',
                        'roles': {'subject': [{'entity_id': 'ent:campaign:alpha',
                                               'kind': 'entity'}]}},
 'payload': {'predicate': 'oc:epistemic_demo_original',
             'roles': {'subject': [{'entity_id': 'ent:campaign:alpha',
                                    'kind': 'entity'}]}},
 'payload_hash': 'sha256:2a62715dfd89b56d18f779df96de0f76815fc5cc34f49b92ed554941f183090a',
 'profile': {'profile_id': 'default', 'profile_version': '1.0.0'},
 'proposal_ids': (),
 'provenance': {'source_artifact': {'content_text': None,
                                    'source_kind': 'note',
                                    'source_label': None,
                                    'source_metadata': {},
                                    'source_ref': 'notes/original.txt'},
                

## Phase 2: Confidence Assessment

Purpose: attach explicit confidence to accepted candidates without changing the
base review schema.

Input -> output: `accepted_candidates -> confidence_state`

Acceptance criteria:
- Confidence is stored through an extension-local package.
- The report can read the resulting confidence assessment.
- The phase runs live.

Status: `proven`
Execution mode: `live`


In [3]:
epistemic_service = EpistemicService(db_path=review_service.store.db_path)
confidence_one = epistemic_service.record_confidence(
    candidate_id=first_candidate.candidate_id,
    confidence_score=0.74,
    source_kind='user',
    actor_id='analyst:confidence',
    rationale='Original reviewed candidate is plausible but broad.',
)
confidence_two = epistemic_service.record_confidence(
    candidate_id=second_candidate.candidate_id,
    confidence_score=0.9,
    source_kind='user',
    actor_id='analyst:confidence',
    rationale='Replacement candidate is narrower and better grounded.',
)
pprint(confidence_one.model_dump())
pprint(confidence_two.model_dump())


{'actor_id': 'analyst:confidence',
 'assessment_id': 'econf_90e814b5cb014b97bfdf5323',
 'candidate_id': 'cand_576cf7b9d22a4479a198dc67',
 'confidence_score': 0.74,
 'created_at': '2026-03-18T17:59:52.912361+00:00',
 'rationale': 'Original reviewed candidate is plausible but broad.',
 'source_kind': 'user'}
{'actor_id': 'analyst:confidence',
 'assessment_id': 'econf_d5e90db2db7a46b586ff19be',
 'candidate_id': 'cand_1198a810705a409194d2591f',
 'confidence_score': 0.9,
 'created_at': '2026-03-18T17:59:52.919069+00:00',
 'rationale': 'Replacement candidate is narrower and better grounded.',
 'source_kind': 'user'}


## Phase 3: Supersession And Report

Purpose: supersede the older accepted candidate with the newer accepted
candidate and expose the result through the report surface.

Input -> output: `confidence_state -> epistemic_extension_proof`

Acceptance criteria:
- Supersession is stored through the extension-local package.
- The report marks the older candidate as superseded.
- The phase runs live.

Status: `proven`
Execution mode: `live`


In [4]:
supersession = epistemic_service.record_supersession(
    prior_candidate_id=first_candidate.candidate_id,
    replacement_candidate_id=second_candidate.candidate_id,
    actor_id='analyst:supersession',
    rationale='Replacement candidate narrows the older accepted claim.',
)
report = EpistemicReportService(epistemic_service=epistemic_service).build_candidate_report(
    candidate_id=first_candidate.candidate_id,
)
assert report.epistemic_status == 'superseded'
assert report.confidence is not None
assert report.superseded_by is not None
assert report.superseded_by.replacement_candidate_id == second_candidate.candidate_id
pprint(supersession.model_dump())
pprint(report.model_dump())


{'actor_id': 'analyst:supersession',
 'created_at': '2026-03-18T17:59:52.931708+00:00',
 'prior_candidate_id': 'cand_576cf7b9d22a4479a198dc67',
 'rationale': 'Replacement candidate narrows the older accepted claim.',
 'replacement_candidate_id': 'cand_1198a810705a409194d2591f',
 'supersession_id': 'esup_feb9efdd1f15462791d60f28'}
{'candidate': {'candidate_id': 'cand_576cf7b9d22a4479a198dc67',
               'claim_text': None,
               'evidence_spans': (),
               'normalized_payload': {'predicate': 'oc:epistemic_demo_original',
                                      'roles': {'subject': [{'entity_id': 'ent:campaign:alpha',
                                                             'kind': 'entity'}]}},
               'payload': {'predicate': 'oc:epistemic_demo_original',
                           'roles': {'subject': [{'entity_id': 'ent:campaign:alpha',
                                                  'kind': 'entity'}]}},
               'payload_hash': 'sha256:2a6271

In [5]:
workspace.cleanup()